# Bremen Crime Data

## Introduction

This project explores published police and news reports associated with Bremen. We will use the English titles and descriptions to investigate report categories, their distribution across districts, and changes over time.

The project questions also cover seasonal peaks, unusual increases, and reports involving elderly people. They are listed on slide 11 of [the course project deck](2026_course_projects_stefano_FINAL.pptx).

The dataset contains publication dates, English text, district labels, source URLs, and two sets of 768-dimensional text embeddings. These embeddings are numerical representations of the text and may help group similar reports later.

Sources: [explanations.html](explanations.html) and bremen_crime_data_2026.parquet. This notebook is the main workstation. Six assignment notebooks supply the preparation, model candidates, reviewed categories and final analyses described below.

## Setup

Run this notebook from the Bremen Crime Data folder. Pandas and pyarrow are already available in the Python environment used for this notebook.

In [1]:
import pandas as pd

### Read the file

In [2]:
crime_data = pd.read_parquet("bremen_crime_data_2026.parquet", engine="pyarrow")

### Convert the publication date

The HTML asks us to convert date_posted into a date type.

In [3]:
crime_data["date_posted"] = pd.to_datetime(crime_data["date_posted"])

### Preview the reports

The preview shows dates, titles, and districts. The full table also retains descriptions, embeddings, and source URLs.

In [4]:
crime_data[["date_posted", "title_en", "district"]].head()

,date_posted,title_en,district
0,2026-09-03,Police release wanted photo after shooting in ...,Woltmershausen
1,2026-09-03,Burglary in Bar - Police Catch Perpetrators,Ostertor
2,2026-09-02,Paint vandalism at high school,Horn
3,2026-09-02,Still waters are deep - and full of e-scooters,Horn
4,2026-09-01,Bogus water workers steal jewelry,Utbremen


In [5]:
print("Rows:", crime_data.shape[0])
print("Columns:", crime_data.shape[1])
print("First publication date:", crime_data["date_posted"].min().date())
print("Last publication date:", crime_data["date_posted"].max().date())

Rows: 5126
Columns: 7
First publication date: 2019-07-14
Last publication date: 2026-09-03


## Initial notes

- The 5,126 reports cover 14 July 2019 to 3 September 2026. The first and last calendar years are incomplete.
- date_posted is the publication date. The incident date may be different.
- There are 273 distinct district labels, including 530 entries containing the text None. We should review district names before comparing areas.
- Two source URLs are repeated. We should inspect those records before deciding whether to remove them.
- Each row represents a report. Counts describe coverage in this dataset, so we should be careful when interpreting them as crime levels or risk.
- There is no separate crime-type or victim-age column. Categories and mentions of elderly people will need to be identified from the text and checked against example reports.

**Current status:** Introduction and loading complete. The group skeleton is ready; modeling, reviewed category labels and final findings are pending.

## Group workstation start here

Read [TEAM_GUIDE.md](TEAM_GUIDE.md), choose one assignment below and agree the group environment. This is the final integration notebook; write your experiments in your own assignment notebook and bring reviewed outputs here.

| Notebook | Responsibility | Main sections supplied |
|---|---|---|
| [01](team_notebooks/01_data_and_districts.ipynb) | Preparation and district analysis | 1 and 6 |
| [02](team_notebooks/02_title_topic_modeling.ipynb) | NMF titles | 2 and comparison inputs |
| [03](team_notebooks/03_description_topic_modeling.ipynb) | NMF descriptions | 3 and comparison inputs |
| [04](team_notebooks/04_embedding_topic_modeling.ipynb) | BERTopic titles/descriptions | 4 and comparison inputs |
| [05](team_notebooks/05_time_and_anomalies.ipynb) | Time/seasonality/anomalies | 7 |
| [06](team_notebooks/06_validation_and_elderly.ipynb) | Validation/selection and elderly victims | 5 and 8 |

**Owners:** 01 ______; 02 ______; 03 ______; 04 ______; 05 ______; 06 ______.  
**Integration coordinator:** ______. Every owner writes their own interpretation and helps review another section.

Dependency order: preparation → four model candidates → reviewed category selection → district/time/elderly analyses → conclusions. Until a handover is accepted, its results are pending rather than zero or complete.

### Shared paths and source version

The base snapshot contains the original rows with permanent IDs. It is not the prepared modeling dataset. The prepared dataset and selected labels will arrive through the handovers.

In [6]:
from pathlib import Path
import json
import hashlib

In [7]:
project_folder = Path.cwd()
assert (project_folder / "shared/source_manifest.json").exists(), "Run the main notebook from the project folder."
with open(project_folder / "shared/source_manifest.json", encoding="utf-8") as file:
    source_manifest = json.load(file)

reports_base = pd.read_parquet(project_folder / "shared/reports_base.parquet", columns=["report_id"])
assert reports_base["report_id"].is_unique
assert len(reports_base) == source_manifest["row_count"]

In [8]:
raw_checksum = hashlib.sha256(Path("bremen_crime_data_2026.parquet").read_bytes()).hexdigest()
assert raw_checksum == source_manifest["raw_sha256"], "The source version changed: resolve this before integrating results."
print("Shared source verified. Base reports:", len(reports_base))

Shared source verified. Base reports: 5126


### Handover inventory

This only checks file presence. Presence does not prove correctness or group approval. Validate schemas, ID sets, versions and evidence in the acceptance section below.

In [9]:
with open("artifact_contracts.json", encoding="utf-8") as file:
    contracts = json.load(file)

handover_status = []
for member, role in contracts["roles"].items():
    found = sum((project_folder / role["output_folder"] / name).exists() for name in role["required_files"])
    handover_status.append({"Member": member, "Files present": found, "Files required": len(role["required_files"])})

pd.DataFrame(handover_status)

,Member,Files present,Files required
0,member_01,0,8
1,member_02,0,6
2,member_03,0,6
3,member_04,0,10
4,member_05,0,5
5,member_06,0,10


## 1. Data preparation

**Owner:** Member 1 early handover.

**Needs:** outputs/member_01/reports_prepared.parquet, data_quality.csv, district_lookup.csv, duplicate_review.csv and metadata/handover.

**What belongs here:** Verify all base report IDs are retained; date_posted is datetime; inclusion flags and exclusion reasons are explicit; original text/embeddings match the source. Record the prepared checksum. Describe geographic level, Unknown handling and duplicate decisions. Do not move to final model runs before this version is accepted.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [10]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 2. Topic modeling using titles

**Owner:** Member 2.

**Needs:** outputs/member_02/nmf_titles_* CSVs, figures and metadata/handover.

**What belongs here:** Explain text preparation, TF-IDF, NMF settings and selected topic count. Import topic keywords, sizes, representative reports and assignments. Check zero-feature/missing/excluded statuses. State strengths and errors; interpretation is part of the submission.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [11]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 3. Topic modeling using descriptions

**Owner:** Member 3.

**Needs:** outputs/member_03/nmf_descriptions_* CSVs, figures and metadata/handover.

**What belongs here:** Use the same prepared version and evaluation report IDs as titles. Explain the description model and compare richer context versus boilerplate/mixed reports. Import evidence and assignments. Do not simply paste a score without explaining it.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [12]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 4. Topic modeling using embeddings

**Owner:** Member 4.

**Needs:** outputs/member_04/bertopic_titles_* and bertopic_descriptions_* CSVs, figures and metadata/handover.

**What belongs here:** Explain supplied embeddings → UMAP → HDBSCAN → topic keywords. Distinguish clustering dimensions from 2D visualization, check both text/embedding pairings, retain -1/unassigned cases, and compare both embedding sources.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [13]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 5. Model comparison and approved categories

**Owner:** Member 6 with all members.

**Needs:** outputs/member_06/review_sample.csv, category_codebook.csv, model_comparison.csv, category_mapping.csv, report_categories.csv, selected_model.json and metadata/handover; category_overrides.csv if used.

**What belongs here:** Compare all four candidates on common held-out report IDs. Review meaningful topics, ambiguity, coverage and stability. Record category definitions, selected model, reasons and approval. Validate complete ID coverage and selected/mapping/prepared checksums before accepting labels. Labels and topic numbers are not interchangeable.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [14]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 6. District distribution and citywide composition

**Owner:** Member 1 later handover.

**Needs:** outputs/member_01/district_category_counts.csv, highest_district_by_category.csv, maps/figures and updated handover.

**What belongs here:** Join accepted categories and prepared records one-to-one. Reconcile counts, show district distribution and citywide composition, retain ties in category leaders, explain denominators and map coverage. Unknown is shown separately; do not report count rankings as population-adjusted risk.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [15]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 7. Time patterns, seasonality and anomalies

**Owner:** Member 5.

**Needs:** outputs/member_05/monthly_counts.csv, seasonal_summary.csv, anomalies.csv, figures and metadata/handover.

**What belongs here:** Use the approved labels and same eligible cohort. Show trends and repeated seasonal patterns, mark partial periods and collection uncertainty, and explain the anomaly rule/baseline. Review underlying reports and distinguish possible causes from evidence-supported explanations.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [16]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 8. Elderly victims

**Owner:** Member 6.

**Needs:** outputs/member_06/elderly_review.csv, elderly_summary.csv, figures and updated metadata/handover.

**What belongs here:** State the elderly definition, search/detection method, victim-context checks and human review. Report yes/no_evidence/unclear/not_reviewed/excluded counts. Answer category/district question using confirmed cases, with evidence excerpts and limitations rather than claiming complete population incidence.

**Acceptance status:** Pending. Import reviewed tables/figures and add a concise interpretation after inspecting their outputs.

In [17]:
# TODO: load and validate this handover using relative paths.
# TODO: display the relevant results and add interpretation below.

**Findings:** Pending.

**Evidence and limitations:** Pending.

## 9. Integration checks and final answers

Before signing off, check the prepared/base/selected-label versions and schemas against artifact_contracts.json. Presence of a file alone is not acceptance.

- Report-level IDs are complete and unique; all model candidates cover all base IDs with explicit statuses.
- Every handover uses the same prepared checksum, and final district/time/elderly results use the same approved category checksum.
- Match topic mappings on model_id and topic_id; record overrides; keep Unknown/unassigned/excluded counts visible.
- Merge prepared reports, final categories and elderly statuses using report_id and validate="one_to_one". A failed join is an error to fix, not a reason to drop records.
- Confirm date types, geographic level, valid category/status values, exclusion reasons and consistent denominators.
- Reconcile group counts against eligible report totals. Explain category-Unknown and district-Unknown separately.
- Read final chart labels/units/date windows, inspect geographic match coverage, and check interpretations against actual outputs.

Write a direct answer with evidence to each question:

| Instructor question | Accepted source section | Answer/evidence |
|---|---|---|
| 1. Categories and district distribution | 2–6 | Pending |
| 2. Trends and seasonal peaks | 7 | Pending |
| 3. Short-period anomalies and possible explanations | 7 | Pending |
| 4. Highest district/category and citywide composition | 6 | Pending |
| 5. Category/district involving elderly victims | 8 | Pending |

**Overall conclusion:** Pending. Keep counts of reports distinct from crime-risk estimates and publication dates distinct from incident dates.

In [18]:
# TODO: assemble the approved final report table and run integration checks.
# Do not fill pending answers until the contributing analyses are reviewed.

## 10. Submission and optional dashboard

- Each member submits their executed notebook, required tables, metadata, figures and handover; another member reviews it.
- Integrate approved results, then restart the kernel and run this main notebook top-to-bottom. Save outputs and inspect every main chart/map at reading size.
- Include setup instructions, tested package versions, method settings, source provenance and limitations so the project can be rerun.
- Check that all five slide-11 questions have supported answers. Do not copy slide 10's made-up example numbers.
- Put the project code on GitHub and invite the instructor as requested on slide 9: Stefanopinto@hotmail.de. Agree repository/data access with the group before the later submission step; this skeleton publishes nothing.
- A static or interactive dashboard is a bonus. Build it from the accepted final table after core analysis is complete.
- The slide deck gives no deadline; confirm the submission date and group milestones.

**Final group sign-off:** 01 ______; 02 ______; 03 ______; 04 ______; 05 ______; 06 ______.  
**Submission status:** Skeleton only — preparation, modeling, reviewed labels, analyses and final conclusions remain pending.